# 2025 VCC Data — Download & Local Access

Downloads the 2025 Virtual Cell Challenge data from Google Cloud Storage and saves it locally for offline use.

**Data source:** Arc Institute GCS bucket — H1 human embryonic stem cells, ~300k cells, 300 genes.

- [Dataset README](https://github.com/ArcInstitute/arc-virtual-cell-atlas/blob/main/virtual-cell-challenge/README.md)
- [tutorial](https://github.com/ArcInstitute/arc-virtual-cell-atlas/blob/main/virtual-cell-challenge/tutorial-py.ipynb)

In [2]:
from pathlib import Path
import pandas as pd
import scanpy as sc
import gcsfs

In [ ]:
DATA_DIR = Path(".")
GCS_BASE = "gs://arc-institute-virtual-cell-atlas/virtual-cell-challenge/2025/"

FILES = {
    "gene_names.csv": "gene_names.csv",
    "pert_counts_Training.csv": "train/pert_counts_Training.csv",
    "adata_Training.h5ad": "train/adata_Training.h5ad",
    "adata_Validation.h5ad": "validation/adata_Validation.h5ad",
}

fs = gcsfs.GCSFileSystem()

## List available files on GCS

Check what's in the bucket — there may be additional files beyond what we've mapped above.

In [9]:
for entry in fs.ls(GCS_BASE, detail=True):
    name = entry["name"].removeprefix("arc-institute-virtual-cell-atlas/virtual-cell-challenge/2025/")
    size_mb = entry.get("size", 0) / 1e6
    print(f"  {name:50s} {size_mb:8.1f} MB")

  FASTQ                                                   0.0 MB
  gene_names.csv                                          0.1 MB
  test                                                    0.0 MB
  train                                                   0.0 MB
  validation                                              0.0 MB


## Download to local

In [ ]:
CHUNK_SIZE = 64 * 1024 * 1024  # 64 MB

for local_name, remote_path in FILES.items():
    local_path = DATA_DIR / local_name
    if local_path.exists():
        print(f"  {local_name} already exists ({local_path.stat().st_size / 1e6:.1f} MB), skipping")
        continue
    remote_full = GCS_BASE + remote_path
    print(f"  Downloading {remote_path} -> {local_name} ...")
    with fs.open(remote_full, "rb") as src, open(local_path, "wb") as dst:
        total = 0
        while chunk := src.read(CHUNK_SIZE):
            dst.write(chunk)
            total += len(chunk)
            print(f"\r    {total / 1e9:.1f} GB", end="", flush=True)
    print(f"\n  Done ({local_path.stat().st_size / 1e6:.1f} MB)")

Could not determine bucket type for bucket name arc-institute-virtual-cell-atlas: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem


    0.0 GB
  Done (0.1 MB)
  pert_counts_Training.csv already exists (0.0 MB), skipping


Could not determine bucket type for bucket name arc-institute-virtual-cell-atlas: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name arc-institute-virtual-cell-atlas: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name arc-institute-virtual-cell-atlas: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem


    0.1 GB

## Load and inspect

In [ ]:
adata_train = sc.read_h5ad(DATA_DIR / "adata_Training.h5ad")
adata_train

NameError: name 'DATA_DIR' is not defined

In [ ]:
pert_counts = pd.read_csv(DATA_DIR / "pert_counts_Training.csv")
print(f"{len(pert_counts)} perturbations")
pert_counts.head(10)

In [ ]:
adata_val = sc.read_h5ad(DATA_DIR / "adata_Validation.h5ad")
adata_val